# 09_dani_ensemble
## Ensemble — Weighted Average of the 12 Hybrid Models

Skema resmi mendaftarkan model Ensemble persis sama dengan Hybrid (`MSTL+UniLSTM, MSTL+BiLSTM,
MSTL+MultivariateLSTM, Prophet+UniLSTM, Prophet+BiLSTM, Prophet+MultiLSTM`), tapi deskripsi input-nya
cuma `"(gunakan hasil tuning dari deep learning)"` — **tanpa** embel-embel "Residual model Utama"
seperti Hybrid. Interpretasi yang dipakai di sini: **Ensemble menggabungkan prediksi dari 12 model
Hybrid yang sudah dilatih** (bukan membangun arsitektur baru), lewat weighted averaging.

### Strategi ensembling

- **Bobot** dihitung dari **validation RMSE** tiap model (10% ekor kronologis dari train — replikasi
  persis `validation_split=0.1, shuffle=False` yang dipakai saat training aslinya), **BUKAN** dari
  performa di test set — supaya bobot ensemble tidak ikut "mengintip" data test (itu akan jadi bentuk
  leakage tersendiri kalau bobotnya dipilih berdasar seberapa bagus performa di test).
- Model dengan validation RMSE lebih kecil dapat bobot lebih besar (`weight ∝ 1/RMSE`, dinormalisasi).
- **Simple average** (bobot sama rata) turut dihitung sebagai pembanding — supaya bisa dilihat apakah
  weighting benar-benar membantu dibanding sekadar rata-rata polos.
- Per-decomposition sub-ensemble (MSTL-only, Prophet-only) turut dihitung sebagai ablasi tambahan.

### Sumber model — TIDAK wajib menunggu notebook 08

`SOURCE` di bawah bisa diarahkan ke:
- `"nb07"` — model default hyperparameter (sudah tersedia sekarang, bisa langsung jalan paralel
  sementara notebook 08 masih tuning).
- `"nb08"` — model hasil tuning per-slot (dipakai begitu notebook 08 selesai; `look_back` per model
  dibaca otomatis dari `08_dani_hybrid_tuned_best_params.csv`, karena tiap slot punya `look_back`
  hasil tuning yang bisa berbeda-beda).

Tidak ada retraining di notebook ini — semua model di-load dari file `.keras` yang sudah tersimpan,
lalu dipakai untuk inference saja (murah secara compute, jauh lebih cepat dari 07/08).

In [ ]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

project_root = next(
    (parent for parent in [Path.cwd(), *Path.cwd().parents] if (parent / "src").exists()),
    Path.cwd()
)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data_loader import TimeSeriesDataLoader

import tensorflow as tf
from prophet import Prophet

print("Project root:", project_root)
print("TensorFlow:", tf.__version__)

In [ ]:
feature_path = (
    project_root / "data" / "processed" / "normalized" / "dataset_feature_engineered.csv"
)

df = pd.read_csv(feature_path)
df["datetime"] = pd.to_datetime(df["datetime"], utc=True)
df = df.set_index("datetime").sort_index()

CI_COL = "carbon_intensity"
RE_COL = "renewable_percentage"

df_core = df.dropna(subset=[CI_COL, RE_COL]).copy()

TRAIN_RATIO = 0.8
train_size = int(len(df_core) * TRAIN_RATIO)
train_raw = df_core.iloc[:train_size].copy()
test_raw = df_core.iloc[train_size:].copy()
TRAIN_END_INDEX = train_raw.index[-1]

print("Train:", train_raw.shape, "| Test:", test_raw.shape)

In [ ]:
TEMPORAL_CYCLICAL = ["hour_sin", "hour_cos", "dow_sin", "dow_cos", "month_sin", "month_cos"]

loader_time = TimeSeriesDataLoader()
all_index_df = pd.DataFrame(index=df_core.index)
cyclical_full = loader_time.add_time_features(all_index_df)[TEMPORAL_CYCLICAL].copy()

print("Cyclical features ready.")

## 1. Decomposition — recomputed fresh (needed for reconstruction, identical to nb07/nb08)

In [ ]:
def compute_causal_decomposition(series, train_end_index, trend_window=168):
    trend = series.rolling(window=trend_window, min_periods=1).mean()
    deseasonalized = series - trend

    train_deseasonalized = deseasonalized.loc[:train_end_index]

    hour_profile = train_deseasonalized.groupby(train_deseasonalized.index.hour).mean()
    seasonal_24 = pd.Series(series.index.hour.map(hour_profile).astype(float), index=series.index)

    residual_after_24 = (
        train_deseasonalized.values
        - train_deseasonalized.index.hour.map(hour_profile).astype(float)
    )
    weekhour_key_train = train_deseasonalized.index.weekday * 24 + train_deseasonalized.index.hour
    weekhour_profile = pd.Series(residual_after_24, index=weekhour_key_train).groupby(level=0).mean()
    weekhour_key_full = series.index.weekday * 24 + series.index.hour
    seasonal_168 = pd.Series(weekhour_key_full.map(weekhour_profile).astype(float), index=series.index)

    resid = series - trend - seasonal_24 - seasonal_168

    return pd.DataFrame(
        {"trend": trend, "seasonal_24": seasonal_24, "seasonal_168": seasonal_168, "resid": resid},
        index=series.index
    )


mstl_ci = compute_causal_decomposition(df_core[CI_COL], TRAIN_END_INDEX, trend_window=168)
mstl_re = compute_causal_decomposition(df_core[RE_COL], TRAIN_END_INDEX, trend_window=168)

mstl_decomposition = {
    "CI_reconstruction": mstl_ci["trend"] + mstl_ci["seasonal_24"] + mstl_ci["seasonal_168"],
    "CI_resid": mstl_ci["resid"],
    "RE_reconstruction": mstl_re["trend"] + mstl_re["seasonal_24"] + mstl_re["seasonal_168"],
    "RE_resid": mstl_re["resid"],
}
print("MSTL decomposition ready.")

In [ ]:
def run_prophet_decomposition(target_col):
    prophet_df = df_core.reset_index()[["datetime", target_col]].copy()
    prophet_df["datetime"] = prophet_df["datetime"].dt.tz_localize(None)
    prophet_df.columns = ["ds", "y"]

    train_prophet_df = prophet_df.iloc[:train_size]
    test_prophet_df = prophet_df.iloc[train_size:]

    model = Prophet(yearly_seasonality=False, weekly_seasonality=True, daily_seasonality=True)
    model.fit(train_prophet_df)

    future = model.make_future_dataframe(periods=len(test_prophet_df), freq="h")
    forecast = model.predict(future)
    assert len(forecast) == len(df_core), "Prophet future length mismatch."

    reconstruction = pd.Series(forecast["yhat"].values, index=df_core.index)
    resid = df_core[target_col] - reconstruction
    return reconstruction, resid


print("Fitting Prophet on CI...")
ci_prophet_reconstruction, ci_prophet_resid = run_prophet_decomposition(CI_COL)
print("Fitting Prophet on RE...")
re_prophet_reconstruction, re_prophet_resid = run_prophet_decomposition(RE_COL)

prophet_decomposition = {
    "CI_reconstruction": ci_prophet_reconstruction,
    "CI_resid": ci_prophet_resid,
    "RE_reconstruction": re_prophet_reconstruction,
    "RE_resid": re_prophet_resid,
}

decompositions = {"mstl": mstl_decomposition, "prophet": prophet_decomposition}
print("Prophet decomposition ready.")

In [ ]:
def build_residual_dataframe(decomp):
    df_resid = pd.DataFrame(index=df_core.index)
    df_resid["CI_resid"] = decomp["CI_resid"]
    df_resid["RE_resid"] = decomp["RE_resid"]
    df_resid[TEMPORAL_CYCLICAL] = cyclical_full[TEMPORAL_CYCLICAL]
    return df_resid


def get_feature_target_cols(target_group):
    if target_group == "CI":
        return ["CI_resid", *TEMPORAL_CYCLICAL], ["CI_resid"]
    elif target_group == "RE":
        return ["RE_resid", *TEMPORAL_CYCLICAL], ["RE_resid"]
    elif target_group == "MULTI":
        return ["CI_resid", "RE_resid", *TEMPORAL_CYCLICAL], ["CI_resid", "RE_resid"]
    raise ValueError("target_group must be CI, RE, or MULTI")


def evaluate(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]

    mae = np.mean(np.abs(y_true - y_pred))
    rmse = np.sqrt(np.mean((y_true - y_pred) ** 2))
    nonzero = y_true != 0
    mape = (
        np.mean(np.abs((y_true[nonzero] - y_pred[nonzero]) / y_true[nonzero])) * 100
        if np.any(nonzero) else np.nan
    )
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    r2 = np.nan if ss_tot == 0 else 1 - ss_res / ss_tot

    return {"MAE": mae, "RMSE": rmse, "MAPE (%)": mape, "R2": r2}


experiment_specs = [
    ("Uni-LSTM CI", "CI", "lstm"),
    ("Uni-LSTM RE", "RE", "lstm"),
    ("Multi-LSTM", "MULTI", "lstm"),
    ("BiLSTM CI", "CI", "bilstm"),
    ("BiLSTM RE", "RE", "bilstm"),
    ("BiLSTM Multi", "MULTI", "bilstm"),
]
decomposition_methods = ["mstl", "prophet"]

## 2. SOURCE toggle — pick which upstream notebook's saved models to ensemble

In [ ]:
SOURCE = "nb07"  # ganti ke "nb08" begitu tuning selesai

TUNED_LOOK_BACK = 12  # dipakai seragam kalau SOURCE == "nb07"

def safe_name(model_name):
    return model_name.lower().replace(" ", "_").replace("-", "")

if SOURCE == "nb07":
    models_dir = project_root / "models" / "hybrid"
    def model_path(method, model_name):
        return models_dir / f"07_hybrid_{method}_{safe_name(model_name)}.keras"
    look_back_lookup = {
        (method, model_name): TUNED_LOOK_BACK
        for method in decomposition_methods
        for model_name, _, _ in experiment_specs
    }

elif SOURCE == "nb08":
    models_dir = project_root / "models" / "hybrid_tuned"
    def model_path(method, model_name):
        return models_dir / f"08_hybrid_tuned_{method}_{safe_name(model_name)}.keras"

    best_params_path = project_root / "results" / "hybrid_tuned" / "08_dani_hybrid_tuned_best_params.csv"
    best_params_df = pd.read_csv(best_params_path)
    look_back_lookup = {
        (row["Decomposition"], row["Model"]): int(row["look_back"])
        for _, row in best_params_df.iterrows()
    }

else:
    raise ValueError("SOURCE must be 'nb07' or 'nb08'")

print("SOURCE:", SOURCE)
print("Models dir:", models_dir)
print("Look-back per config:", look_back_lookup)

## 3. Load all 12 base models + compute validation RMSE (for ensemble weights)

In [ ]:
VALIDATION_SPLIT = 0.1
FORECAST_HORIZON = 1

def get_validation_arrays(X, y, validation_split=VALIDATION_SPLIT):
    n_val = max(1, int(len(X) * validation_split))
    return X[-n_val:], y[-n_val:]


base_models = {}

for method in decomposition_methods:
    decomp = decompositions[method]
    df_resid = build_residual_dataframe(decomp)

    for model_name, target_group, architecture in experiment_specs:
        look_back = look_back_lookup[(method, model_name)]
        feature_cols, target_cols = get_feature_target_cols(target_group)

        path = model_path(method, model_name)
        if not path.exists():
            print(f"WARNING: missing model file, skipped: {path}")
            continue

        model = tf.keras.models.load_model(path)

        loader = TimeSeriesDataLoader()
        train_scaled, test_scaled = loader.split_and_scale(
            df=df_resid, feature_cols=feature_cols, target_cols=target_cols, train_ratio=TRAIN_RATIO
        )
        X_train, y_train = loader.create_sliding_window(
            train_scaled, target_cols=target_cols, look_back=look_back, forecast_horizon=FORECAST_HORIZON
        )
        X_test, y_test = loader.create_sliding_window(
            test_scaled, target_cols=target_cols, look_back=look_back, forecast_horizon=FORECAST_HORIZON
        )
        X_val, y_val = get_validation_arrays(X_train, y_train)

        # --- validation RMSE per target this model contributes to (raw scale) ---
        val_pred_scaled = model.predict(X_val, verbose=0)
        if len(target_cols) == 1:
            val_pred_scaled = val_pred_scaled.reshape(-1)

        val_pred_inv = loader.inverse_transform_predictions(val_pred_scaled, target_cols, feature_cols)
        val_actual_inv = loader.inverse_transform_predictions(y_val, target_cols, feature_cols)

        train_window_index = train_scaled.index[look_back:]
        val_index = train_window_index[-len(y_val):]

        val_rmse = {}
        for i, col in enumerate(target_cols):
            target_name = "CI" if col.startswith("CI") else "RE"
            recon = decomp[f"{target_name}_reconstruction"].loc[val_index].values
            if len(target_cols) == 1:
                pred_final = val_pred_inv.reshape(-1) + recon
            else:
                pred_final = val_pred_inv[:, i] + recon
            actual_raw = df_core.loc[val_index, CI_COL if target_name == "CI" else RE_COL].values
            val_rmse[target_name] = np.sqrt(np.mean((actual_raw - pred_final) ** 2))

        # --- one-step test predictions (raw scale), aligned to this model's own valid window ---
        test_pred_scaled = model.predict(X_test, verbose=0)
        if len(target_cols) == 1:
            test_pred_scaled = test_pred_scaled.reshape(-1)

        test_pred_inv = loader.inverse_transform_predictions(test_pred_scaled, target_cols, feature_cols)
        own_test_index = test_raw.index[look_back:]

        preds_by_target = {}
        for i, col in enumerate(target_cols):
            target_name = "CI" if col.startswith("CI") else "RE"
            recon = decomp[f"{target_name}_reconstruction"].loc[own_test_index].values
            if len(target_cols) == 1:
                final_pred = test_pred_inv.reshape(-1) + recon
            else:
                final_pred = test_pred_inv[:, i] + recon
            preds_by_target[target_name] = pd.Series(final_pred, index=own_test_index)

        base_models[(method, model_name)] = {
            "model": model,
            "feature_cols": feature_cols,
            "target_cols": target_cols,
            "target_group": target_group,
            "look_back": look_back,
            "val_rmse": val_rmse,
            "test_pred": preds_by_target,
        }

        print(f"Loaded: {model_name} | {method} | look_back={look_back} | val_rmse={val_rmse}")

print("\nTotal base models loaded:", len(base_models))

## 4. One-step ensemble — simple average & inverse-RMSE weighted average

Semua model diselaraskan ke jendela test yang sama (`MAX_LOOK_BACK` = look-back terpanjang di antara
semua model, supaya SEMUA model punya prediksi valid di setiap titik jendela ini).

In [ ]:
MAX_LOOK_BACK = max(bundle["look_back"] for bundle in base_models.values())
common_test_index = test_raw.index[MAX_LOOK_BACK:]

print("MAX_LOOK_BACK:", MAX_LOOK_BACK)
print("Common test window:", len(common_test_index), "rows")


def gather_target_predictions(target_name, method_filter=None):
    rows = []
    for (method, model_name), bundle in base_models.items():
        if method_filter is not None and method != method_filter:
            continue
        if target_name not in bundle["test_pred"]:
            continue
        pred = bundle["test_pred"][target_name].reindex(common_test_index)
        rows.append({
            "key": f"{method}_{model_name}",
            "pred": pred,
            "weight_raw": 1.0 / bundle["val_rmse"][target_name]
        })
    return rows


def combine_predictions(rows, weighted):
    if weighted:
        total_w = sum(r["weight_raw"] for r in rows)
        weights = [r["weight_raw"] / total_w for r in rows]
    else:
        weights = [1.0 / len(rows)] * len(rows)

    combined = sum(w * r["pred"] for w, r in zip(weights, rows))
    return combined, {r["key"]: w for w, r in zip(weights, rows)}


ensemble_results = []
ensemble_predictions = {}

for target_name in ["CI", "RE"]:
    actual = df_core.loc[common_test_index, CI_COL if target_name == "CI" else RE_COL].values

    for label, method_filter in [("all", None), ("mstl_only", "mstl"), ("prophet_only", "prophet")]:
        rows = gather_target_predictions(target_name, method_filter=method_filter)
        if len(rows) == 0:
            continue

        for weighted, tag in [(False, "Ensemble-Simple"), (True, "Ensemble-Weighted")]:
            combined, weights = combine_predictions(rows, weighted=weighted)
            metrics = evaluate(actual, combined.values)
            ensemble_results.append({
                "Ensemble": f"{tag} ({label})",
                "Target": target_name,
                "N_models": len(rows),
                **metrics
            })
            if label == "all":
                ensemble_predictions[(tag, target_name)] = combined

ensemble_results_df = pd.DataFrame(ensemble_results)
print("=== ONE-STEP ENSEMBLE EVALUATION ===")
display(ensemble_results_df.sort_values(["Target", "RMSE"]).reset_index(drop=True))

In [ ]:
# Compare best ensemble vs best individual base model, per target.
individual_results = []
for (method, model_name), bundle in base_models.items():
    for target_name, pred in bundle["test_pred"].items():
        pred_aligned = pred.reindex(common_test_index)
        actual = df_core.loc[common_test_index, CI_COL if target_name == "CI" else RE_COL].values
        metrics = evaluate(actual, pred_aligned.values)
        individual_results.append({
            "Model": f"{method}_{model_name}", "Target": target_name, **metrics
        })

individual_results_df = pd.DataFrame(individual_results)

for target_name in ["CI", "RE"]:
    best_individual = individual_results_df[individual_results_df["Target"] == target_name].sort_values("RMSE").iloc[0]
    best_ensemble = ensemble_results_df[ensemble_results_df["Target"] == target_name].sort_values("RMSE").iloc[0]

    print(f"--- {target_name} ---")
    print(f"Best individual base model : {best_individual['Model']:30s} RMSE={best_individual['RMSE']:.4f}")
    print(f"Best ensemble               : {best_ensemble['Ensemble']:30s} RMSE={best_ensemble['RMSE']:.4f}")
    improvement = (best_individual["RMSE"] - best_ensemble["RMSE"]) / best_individual["RMSE"] * 100
    print(f"Improvement: {improvement:.2f}%\n")

## 5. Multistep ensemble — rolling-origin, shared origins across all base models

Origin dihitung sekali dari `MAX_LOOK_BACK` dan dipakai SAMA untuk semua model, supaya prediksi tiap
model bisa diselaraskan (per origin, per horizon-step) sebelum digabung.

In [ ]:
MULTISTEP_HORIZON = 24
ORIGIN_STRIDE = 24
N_ORIGINS_MAX = 100


def rolling_origin_recursive_forecast(model, loader, test_scaled, feature_cols, target_cols, horizon, look_back, origins):
    data = test_scaled[feature_cols].values
    target_idx = [feature_cols.index(c) for c in target_cols]

    per_origin_preds = []
    for origin in origins:
        history_window = data[origin - look_back: origin].copy()
        preds_scaled = []

        for h in range(horizon):
            X_step = history_window[np.newaxis, :, :]
            pred_scaled = model.predict(X_step, verbose=0).reshape(-1)
            preds_scaled.append(pred_scaled)

            next_row = data[origin + h].copy()
            for i, idx in enumerate(target_idx):
                next_row[idx] = pred_scaled[i]
            history_window = np.vstack([history_window[1:], next_row])

        preds_scaled = np.array(preds_scaled)
        preds_inv = loader.inverse_transform_predictions(preds_scaled, target_cols=target_cols, feature_cols=feature_cols)
        per_origin_preds.append((origin, preds_inv))

    return per_origin_preds


# Shared origins - valid for every model regardless of its own (possibly smaller) look_back.
sample_test_scaled_len = len(test_raw)  # same length across all configs (same df_core/test split)
shared_origins = list(range(MAX_LOOK_BACK, sample_test_scaled_len - MULTISTEP_HORIZON, ORIGIN_STRIDE))[:N_ORIGINS_MAX]

print("Shared origins:", len(shared_origins))

In [ ]:
# Re-run rolling-origin recursive forecast per base model (cheap: inference only, no retraining).
base_multistep_preds = {}

for (method, model_name), bundle in base_models.items():
    decomp = decompositions[method]
    df_resid = build_residual_dataframe(decomp)
    feature_cols, target_cols = bundle["feature_cols"], bundle["target_cols"]

    loader = TimeSeriesDataLoader()
    _, test_scaled = loader.split_and_scale(df_resid, feature_cols, target_cols, train_ratio=TRAIN_RATIO)

    per_origin_preds = rolling_origin_recursive_forecast(
        bundle["model"], loader, test_scaled, feature_cols, target_cols,
        horizon=MULTISTEP_HORIZON, look_back=bundle["look_back"], origins=shared_origins
    )

    # store as {target_name: DataFrame[origin_idx, horizon_step] = raw-scale prediction}
    per_target_grid = {t: {} for t in ["CI", "RE"] if t in [("CI" if c.startswith("CI") else "RE") for c in target_cols]}

    for origin, preds_inv in per_origin_preds:
        preds_inv = np.atleast_2d(preds_inv)
        if preds_inv.shape[0] == 1 and len(target_cols) > 1:
            preds_inv = preds_inv.reshape(-1, len(target_cols))
        elif preds_inv.ndim == 1:
            preds_inv = preds_inv.reshape(-1, 1)

        origin_index = test_scaled.index[origin: origin + MULTISTEP_HORIZON]

        for i, col in enumerate(target_cols):
            target_name = "CI" if col.startswith("CI") else "RE"
            recon_vals = decomp[f"{target_name}_reconstruction"].loc[origin_index].values
            final_pred = preds_inv[:, i] + recon_vals
            per_target_grid[target_name][origin] = final_pred

    base_multistep_preds[(method, model_name)] = per_target_grid
    print(f"Multistep done: {model_name} | {method}")

In [ ]:
# Combine multistep predictions across contributing models (simple & weighted), per origin/horizon.
ensemble_horizon_results = []

for target_name in ["CI", "RE"]:
    contributing = [
        (key, bundle) for key, bundle in base_models.items()
        if target_name in base_multistep_preds[key]
    ]

    for weighted, tag in [(False, "Ensemble-Simple"), (True, "Ensemble-Weighted")]:
        if weighted:
            total_w = sum(1.0 / b["val_rmse"][target_name] for _, b in contributing)
            weights = {key: (1.0 / b["val_rmse"][target_name]) / total_w for key, b in contributing}
        else:
            weights = {key: 1.0 / len(contributing) for key, b in contributing}

        for origin in shared_origins:
            combined = np.zeros(MULTISTEP_HORIZON)
            for key, _ in contributing:
                combined += weights[key] * base_multistep_preds[key][target_name][origin]

            origin_index = test_raw.index[origin: origin + MULTISTEP_HORIZON]
            actual = df_core.loc[origin_index, CI_COL if target_name == "CI" else RE_COL].values

            for h in range(MULTISTEP_HORIZON):
                ensemble_horizon_results.append({
                    "Ensemble": tag, "Target": target_name,
                    "Horizon": h + 1, "AbsError": abs(actual[h] - combined[h])
                })

ensemble_horizon_df = pd.DataFrame(ensemble_horizon_results)
ensemble_horizon_summary = (
    ensemble_horizon_df
    .groupby(["Ensemble", "Target", "Horizon"], as_index=False)["AbsError"]
    .mean()
    .rename(columns={"AbsError": "MAE_at_horizon"})
)

print("=== MULTISTEP ENSEMBLE MAE BY HORIZON ===")
display(ensemble_horizon_summary.head(10))

In [ ]:
for target_name in ["CI", "RE"]:
    fig, ax = plt.subplots(figsize=(10, 6))
    for tag in ["Ensemble-Simple", "Ensemble-Weighted"]:
        subset = ensemble_horizon_summary[
            (ensemble_horizon_summary["Ensemble"] == tag) & (ensemble_horizon_summary["Target"] == target_name)
        ].sort_values("Horizon")
        ax.plot(subset["Horizon"], subset["MAE_at_horizon"], label=tag, marker="o", markersize=3)

    ax.set_xlabel("Horizon (jam ke depan)")
    ax.set_ylabel("MAE")
    ax.set_title(f"{target_name} — Ensemble MAE vs horizon")
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
plot_index = common_test_index

fig, axes = plt.subplots(2, 1, figsize=(15, 10), sharex=True)

axes[0].plot(plot_index, df_core.loc[plot_index, CI_COL].values, label="Actual CI", linewidth=1)
axes[0].plot(plot_index, ensemble_predictions[("Ensemble-Weighted", "CI")].values, label="Ensemble-Weighted CI", linewidth=1)
axes[0].set_title("CI — Ensemble (weighted, all 8 contributing models)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(plot_index, df_core.loc[plot_index, RE_COL].values, label="Actual RE", linewidth=1)
axes[1].plot(plot_index, ensemble_predictions[("Ensemble-Weighted", "RE")].values, label="Ensemble-Weighted RE", linewidth=1)
axes[1].set_title("RE — Ensemble (weighted, all 8 contributing models)")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
results_dir = project_root / "results" / "ensemble"
results_dir.mkdir(parents=True, exist_ok=True)

ensemble_results_df.to_csv(results_dir / f"09_dani_ensemble_onestep_metrics_{SOURCE}.csv", index=False)
individual_results_df.to_csv(results_dir / f"09_dani_ensemble_individual_base_metrics_{SOURCE}.csv", index=False)
ensemble_horizon_summary.to_csv(results_dir / f"09_dani_ensemble_multistep_horizon_mae_{SOURCE}.csv", index=False)

print("Saved results to:", results_dir, f"(SOURCE={SOURCE})")

## Ringkasan

- **`ensemble_results_df`** — one-step: Simple vs Weighted average, gabungan semua model (`all`) dan
  sub-ensemble per decomposition method (`mstl_only`, `prophet_only`).
- **`individual_results_df`** — performa tiap model individu di jendela test yang sama, buat baseline
  perbandingan langsung ke ensemble.
- **`ensemble_horizon_summary`** — kurva MAE-vs-horizon untuk kedua strategi ensembling.

Yang perlu diperiksa:
1. Apakah ensemble (simple ATAU weighted) benar-benar mengungguli model individu terbaik — cek print
   "Improvement (%)" di atas. Kalau minus/negatif, itu insight valid juga: berarti model terbaik individu
   sudah cukup dominan, dan menggabungkan model yang lebih lemah malah menariknya turun.
2. Apakah weighted mengungguli simple average secara konsisten, atau tidak — kalau bedanya tipis,
   simple average lebih defensible untuk dilaporkan (lebih sederhana, tidak butuh penjelasan skema bobot).
3. Ganti `SOURCE = "nb08"` begitu tuning selesai, rerun, bandingkan apakah ensemble dari model yang
   sudah di-tuning memberi hasil yang jauh lebih baik dari ensemble model default (nb07).